# Cleaning and checking HESA student enrolment data

This notebook turns HESA's [Table 1: HE student enrolments by HE provider](https://www.hesa.ac.uk/data-and-analysis/students/table-1) into the verified dataset behind the dashboard. The data is used under the [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) licence.

It works through seven steps:

1. Read HESA's files, which have information rows above the data
2. Load all the years, keeping only the rows needed
3. Remove the UK total row, which would otherwise double count every student
4. Check every year against HESA's own published total
5. Check that each breakdown adds up to the total
6. Build and save the clean dataset
7. Export the data the dashboard reads

**The checks in steps 4 to 6 use `assert`**, so if the numbers ever stop adding up (for example after downloading a new year of data), the notebook stops with an explanation instead of quietly producing a wrong dashboard.

To run it, download the source data zip from the HESA page, extract the CSV files into `data/raw/`, and click **Run All**.

In [1]:
import json
import re
from datetime import date, datetime
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
DASHBOARD = Path("../dashboard")

files = sorted(RAW.glob("table-1-*.csv"))
assert files, "No HESA files found in data/raw. Download and extract them first (see the README)."
print(f"{len(files)} files found, from {files[0].name} to {files[-1].name}")

11 files found, from table-1-(2014-15).csv to table-1-(2024-25).csv


## 1. Reading HESA's files

Each file starts with information rows (title, licence, HESA's published total, the date it was last updated) before the column headings. The number of those rows could differ between years, so rather than assuming the headings are always on the same line, `find_header` looks for the line that starts with `UKPRN`.

The other two functions read HESA's published total and last-updated date from the top of each file. The total is used to check the data in step 4, and the date is shown on the dashboard.

In [2]:
def find_header(path):
    """Return the line number where the real column headings start."""
    with open(path, encoding="utf-8-sig") as f:
        for i, line in enumerate(f):
            if line.startswith("UKPRN"):
                return i
    raise ValueError(f"No header row found in {path}")


def published_total(path):
    """Return (academic year, HESA's published total) from a line like '2024/25 total,2863180'."""
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            first = line.split(",")[0]
            if first.endswith(" total"):
                year = first.replace(" total", "")
                return year, int(line.split(",")[1].strip('"').replace(",", ""))
    raise ValueError(f"No published total found in {path}")


def hesa_last_updated(path):
    """Return the date HESA last updated the file, from a line like 'Last updated,Jan-26'."""
    with open(path, encoding="utf-8-sig") as f:
        for line in f:
            if line.startswith("Last updated"):
                value = line.split(",")[1].strip().strip('"')
                return datetime.strptime(value, "%b-%y").strftime("%B %Y")
    return None


print("Headings start on line", find_header(files[-1]), "of", files[-1].name)
print("Published total:", published_total(files[-1]))
print("HESA last updated:", hesa_last_updated(files[-1]))

Headings start on line 13 of table-1-(2024-25).csv
Published total: ('2024/25', 2863180)
HESA last updated: January 2026


## 2. Loading all the years

Each file has around 800,000 rows, nearly 2 million across all years, which is more than Excel can open. Most of those rows repeat the same students at different levels of detail, so each file is cut down as it loads:

- **Country and region of provider:** each university appears once with these set to `All`, and again under its own country and region. Keeping only the `All` rows avoids counting it two or three times.
- **Numbers** are stored as text with commas (`"13,680"`), so they're converted to numbers.
- **Some category names have stray spaces** (`"Total UK  "`), so they're trimmed.

In [3]:
def load_year(path):
    df = pd.read_csv(path, skiprows=find_header(path), dtype=str)
    df = df[
        (df["Country of HE provider"] == "All") &
        (df["Region of HE provider"] == "All")
    ].copy()
    df["Category"] = df["Category"].str.strip()
    df["Number"] = pd.to_numeric(df["Number"].str.replace(",", ""), errors="coerce")
    return df


raw = pd.concat([load_year(p) for p in files], ignore_index=True)
print(f"{len(raw):,} rows loaded across {raw['Academic Year'].nunique()} years")

for col in ["Entrant marker", "Level of study", "Mode of study", "Category marker"]:
    print(f"{col}: {raw[col].unique().tolist()}")

1,989,036 rows loaded across 11 years
Entrant marker: ['All', 'Entrant', 'Not an entrant']
Level of study: ['All', 'Postgraduate (research)', 'Postgraduate (taught)', 'All postgraduate', 'First degree', 'Other undergraduate', 'All undergraduate']
Mode of study: ['All', 'Full-time', 'Part-time']
Category marker: ['Sex', 'Permanent address', 'Total']


## 3. Removing the UK total row

Every breakdown in HESA's file includes subtotals alongside the detail: `All` rows, and levels such as `All undergraduate` that are the sum of other levels. To get one overall figure per provider, every breakdown has to be set to `All` at once.

Even then, there's a second trap. HESA includes a row where the provider is `Total`, the whole UK figure, listed as if it were a university. **My first attempt included it and came out at almost exactly double the real total.** The cell below shows that, then removes the row.

In [4]:
def overall_totals(df):
    """One overall figure per provider per year: every breakdown set to 'All'."""
    return df[
        (df["Entrant marker"] == "All") &
        (df["Level of study"] == "All") &
        (df["Mode of study"] == "All") &
        (df["Category marker"] == "Total")
    ]


published = dict(published_total(p) for p in files)
latest_year = max(published)

with_total_row = overall_totals(raw).query("`Academic Year` == @latest_year")["Number"].sum()
print(f"{latest_year} including the 'Total' row: {with_total_row:,} "
      f"({with_total_row / published[latest_year]:.2f} times HESA's published total)")

all_years = raw[raw["HE provider"] != "Total"].copy()
without = overall_totals(all_years).query("`Academic Year` == @latest_year")["Number"].sum()
print(f"{latest_year} without it: {without:,} (HESA's published total: {published[latest_year]:,})")

2024/25 including the 'Total' row: 5,726,355 (2.00 times HESA's published total)
2024/25 without it: 2,863,175 (HESA's published total: 2,863,180)


## 4. Checking every year against HESA's published total

HESA rounds every figure to the nearest 5 to protect individual students, so adding up around 300 rounded figures won't exactly match the published total. A difference of a few dozen is expected. **Anything over 50 would mean students are being double counted or missed**, so the notebook stops.

The count of providers each year is saved too. It shows how HESA's coverage has grown, which the dashboard explains to users.

In [5]:
MAX_DIFFERENCE = 50

check = overall_totals(all_years).groupby("Academic Year")["Number"].agg(providers="size", our_total="sum")
check["published"] = pd.Series(published)
check["difference"] = check["our_total"] - check["published"]
display(check)

too_far = check[check["difference"].abs() > MAX_DIFFERENCE]
assert too_far.empty, f"These years differ from HESA's published total by more than {MAX_DIFFERENCE}:\n{too_far}"
print(f"All {len(check)} years match HESA's published totals to within {check['difference'].abs().max()} students.")

,providers,our_total,published,difference
Academic Year,,,,
2014/15,226,2315800,2315840,-40
2015/16,261,2332825,2332825,0
2016/17,262,2378020,2378020,0
2017/18,266,2415300,2415335,-35
2018/19,266,2457285,2457250,35
2019/20,271,2529850,2529870,-20
2020/21,282,2747200,2747200,0
2021/22,285,2857835,2857855,-20
2022/23,291,2937260,2937285,-25


All 11 years match HESA's published totals to within 40 students.


## 5. Checking that each breakdown adds up

The dashboard splits students three ways. For each one, only categories that don't overlap are used, so they should add up to the total, within rounding:

- **Level of study:** the four detailed levels, not the `All postgraduate` or `All undergraduate` subtotals
- **Where students come from:** `Total UK`, `European Union`, `Non-European Union` and `Not known`, not `Total Non-UK` (which is EU plus non-EU) or the individual UK nations (which make up `Total UK`)
- **Mode of study:** full-time and part-time

Rounding means these won't match exactly, but double counting would put them out by 100% or more, so a 1% tolerance catches any real problem.

In [6]:
LEVELS = ["Postgraduate (research)", "Postgraduate (taught)", "First degree", "Other undergraduate"]
DOMICILES = ["Total UK", "European Union", "Non-European Union", "Not known"]
MODES = ["Full-time", "Part-time"]
TOLERANCE = 0.01  # 1%

base = all_years[all_years["Entrant marker"] == "All"]
breakdowns = {
    "level": base[(base["Mode of study"] == "All") & (base["Category marker"] == "Total") & base["Level of study"].isin(LEVELS)],
    "where from": base[(base["Level of study"] == "All") & (base["Mode of study"] == "All")
                       & (base["Category marker"] == "Permanent address") & base["Category"].isin(DOMICILES)],
    "mode": base[(base["Level of study"] == "All") & (base["Category marker"] == "Total") & base["Mode of study"].isin(MODES)],
}

sums = pd.DataFrame({name: rows.groupby("Academic Year")["Number"].sum() for name, rows in breakdowns.items()})
gaps = sums.div(check["our_total"], axis=0).sub(1)
display(gaps.style.format("{:+.3%}"))

assert (gaps.abs() <= TOLERANCE).all().all(), f"A breakdown is more than {TOLERANCE:.0%} away from the total"
print(f"Every breakdown adds up to within {gaps.abs().max().max():.3%} of the total in every year.")

,level,where from,mode
Academic Year,,,
2014/15,+0.002%,+0.004%,+0.001%
2015/16,-0.001%,-0.003%,+0.001%
2016/17,+0.000%,-0.001%,+0.000%
2017/18,+0.003%,+0.003%,+0.001%
2018/19,+0.001%,-0.004%,+0.001%
2019/20,-0.002%,+0.001%,-0.002%
2020/21,+0.002%,+0.001%,-0.000%
2021/22,+0.000%,+0.000%,-0.001%
2022/23,-0.001%,-0.003%,+0.001%


Every breakdown adds up to within 0.004% of the total in every year.


## 6. Building the clean dataset

This pulls out the five views the dashboard uses, using exactly the filters checked above, and saves them as one tidy table: one row per provider, year, measure and group.

Providers are matched on their **UKPRN**, their official ID, rather than their name. Universities sometimes rename themselves, and matching on the ID keeps each one's history together under its latest name.

In [7]:
d = all_years.rename(columns={
    "UKPRN": "ukprn", "HE provider": "provider", "Academic Year": "year",
    "Entrant marker": "entrant", "Level of study": "level",
    "Mode of study": "mode", "Category marker": "marker",
    "Category": "category", "Number": "number",
})

def is_all(col):
    return d[col] == "All"

is_total = d["marker"] == "Total"

def section(name, mask, group_col=None):
    out = d[mask].copy()
    out["measure"] = name
    out["group"] = out[group_col] if group_col else "All"
    return out[["ukprn", "year", "measure", "group", "number"]]

clean = pd.concat([
    section("total", is_all("entrant") & is_all("level") & is_all("mode") & is_total),
    section("level", is_all("entrant") & is_all("mode") & is_total & d["level"].isin(LEVELS), "level"),
    section("mode", is_all("entrant") & is_all("level") & is_total & d["mode"].isin(MODES), "mode"),
    section("domicile", is_all("entrant") & is_all("level") & is_all("mode")
            & (d["marker"] == "Permanent address") & d["category"].isin(DOMICILES), "category"),
    section("entrants", (d["entrant"] == "Entrant") & is_all("level") & is_all("mode") & is_total),
], ignore_index=True)

clean["group"] = clean["group"].replace({"Total UK": "UK"})

# Use each provider's most recent name, in case it changed over the years
latest_names = d.sort_values("year").groupby("ukprn")["provider"].last()
clean["provider"] = clean["ukprn"].map(latest_names)

duplicates = clean.duplicated(["ukprn", "year", "measure", "group"]).sum()
assert duplicates == 0, f"{duplicates} duplicate rows: two filters are overlapping"

yearly = clean[clean["measure"] == "total"].groupby("year")["number"].sum()
assert (yearly == check["our_total"]).all(), "The clean totals don't match the checked totals"

PROCESSED.mkdir(parents=True, exist_ok=True)
clean.to_csv(PROCESSED / "enrolments_clean.csv", index=False)
print(f"Saved {len(clean):,} rows for {clean['ukprn'].nunique()} providers, with no duplicates.")
clean.head()

Saved 32,480 rows for 358 providers, with no duplicates.


,ukprn,year,measure,group,number,provider
0,10007783,2014/15,total,All,14035,The University of Aberdeen
1,10019746,2014/15,total,All,110,ABI College Limited
2,10007849,2014/15,total,All,4220,Abertay University
3,10007856,2014/15,total,All,9835,Aberystwyth University
4,10000080,2014/15,total,All,70,Access to Music Limited


## 7. Exporting for the dashboard

The dashboard reads one JSON file. For each provider, it holds every measure as a list of 11 values, one per year, with gaps where HESA has no figures.

It also records the details the dashboard displays, so none of them need typing in by hand when new data is added:

- the date HESA last updated the data, read from the file itself
- the date this export was run
- the number of providers in HESA's data each year

In [8]:
years = sorted(clean["year"].unique())

def display_name(name):
    # "The University of Manchester" -> "University of Manchester", so the search list sorts sensibly
    return re.sub(r"^The ", "", name)

today = date.today()
output = {
    "years": years,
    "hesa_updated": hesa_last_updated(files[-1]),
    "built": f"{today.day} {today:%B %Y}",
    "provider_counts": {year: int(n) for year, n in check["providers"].items()},
    "providers": [],
    "data": {},
}

for ukprn, group in clean.groupby("ukprn"):
    entry = {}
    for measure, rows in group.groupby("measure"):
        table = rows.pivot_table(index="group", columns="year", values="number", aggfunc="sum")
        table = table.reindex(columns=years)  # every provider gets every year, with gaps as None
        series = {
            name: [None if pd.isna(v) else int(v) for v in values]
            for name, values in table.iterrows()
        }
        entry[measure] = series.get("All") if measure in ("total", "entrants") else series
    output["data"][str(ukprn)] = entry
    output["providers"].append({"id": str(ukprn), "name": display_name(group["provider"].iloc[0])})

output["providers"].sort(key=lambda p: p["name"])

DASHBOARD.mkdir(parents=True, exist_ok=True)
with open(DASHBOARD / "data.json", "w", encoding="utf-8") as f:
    json.dump(output, f, separators=(",", ":"))

print("Providers exported:", len(output["providers"]))
print("HESA last updated:", output["hesa_updated"])
print("Dashboard built:", output["built"])

Providers exported: 358
HESA last updated: January 2026
Dashboard built: 24 September 2026
